# 01 - Data Understanding

First pass over the raw Olist tables before touching anything: how big is
each table, what does the schema look like, and how much of the `orders`
table can we actually use to build a delay label.

In [1]:
import sys
sys.path.append("../src")
import pandas as pd
from preprocessing import load_raw_tables

pd.set_option("display.max_columns", 50)

orders, items, customers, sellers, products, translation, payments, geo = load_raw_tables()

tables = {
    "orders": orders, "order_items": items, "customers": customers,
    "sellers": sellers, "products": products,
    "category_translation": translation, "payments": payments, "geolocation": geo,
}
for name, t in tables.items():
    print(f"{name:22s} rows={len(t):>9,}  cols={t.shape[1]}")

orders                 rows=   99,441  cols=8
order_items            rows=  112,650  cols=7
customers              rows=   99,441  cols=5
sellers                rows=    3,095  cols=4
products               rows=   32,951  cols=9
category_translation   rows=       71  cols=2
payments               rows=  103,886  cols=5
geolocation            rows=1,000,163  cols=5


## Orders table - the backbone of the whole project
This is the only table that carries `order_estimated_delivery_date` and
`order_delivered_customer_date` side by side, which is what makes a
genuine "was this late" label possible.

In [2]:
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [3]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Only `delivered` orders have a real delivered-customer-date, so the delay
label can only be constructed for that subset. Everything else
(`canceled`, `unavailable`, `shipped`, ...) will be kept around for the
operational overview charts but excluded from the supervised learning
problem itself.

In [4]:
orders["order_delivered_customer_date"].isna().groupby(orders["order_status"]).mean()

order_status
approved       1.000000
canceled       0.990400
created        1.000000
delivered      0.000083
invoiced       1.000000
processing     1.000000
shipped        1.000000
unavailable    1.000000
Name: order_delivered_customer_date, dtype: float64

## Missing values across the raw tables

In [5]:
for name, t in tables.items():
    miss = t.isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    if len(miss):
        print(f"\n{name}:")
        print((miss * 100).round(2).astype(str) + "%")


orders:
order_delivered_customer_date    2.98%
order_delivered_carrier_date     1.79%
order_approved_at                0.16%
dtype: str

products:
product_category_name         1.85%
product_description_lenght    1.85%
product_name_lenght           1.85%
product_photos_qty            1.85%
product_weight_g              0.01%
product_height_cm             0.01%
product_length_cm             0.01%
product_width_cm              0.01%
dtype: str


`order_products` has some missing category / dimension fields, and a
small slice of `orders` is missing `order_approved_at` /
`order_delivered_carrier_date` - both get handled explicitly in the
cleaning + feature engineering steps rather than silently dropped.